In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

#plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

In [4]:
df = pd.read_csv('steam_indie_merged_data.csv', engine='python')

In [5]:
df['name_store'].to_list()

['Palworld',
 'Unturned',
 'Terraria',
 'Wallpaper Engine',
 'Brawlhalla',
 "Garry's Mod",
 'Rust',
 'ARK: Survival Evolved',
 'Stardew Valley',
 'The Forest',
 'Last Epoch',
 'Fall Guys',
 'Euro Truck Simulator 2',
 'Valheim',
 'Phasmophobia',
 'Heroes & Generals',
 'Rocket League®',
 'Project Zomboid',
 'Lethal Company',
 'Raft',
 'Schedule I',
 "Don't Starve Together",
 'Human Fall Flat',
 '7 Days to Die',
 'CyberCorp',
 'Sons Of The Forest',
 'Grim Dawn',
 'Mount & Blade II: Bannerlord',
 'Warhammer 40,000: Rogue Trader',
 'Satisfactory',
 'Risk of Rain 2',
 'Castle Crashers®',
 'Geometry Dash',
 'No More Room in Hell',
 'Hollow Knight',
 'Robocraft',
 'Deceit',
 'A Story About My Uncle',
 'Subnautica',
 'Kathy Rain',
 'Guacamelee! Super Turbo Championship Edition',
 'Content Warning',
 'ULTRAKILL',
 'Chivalry: Medieval Warfare',
 'Assetto Corsa',
 'Vampire Survivors',
 'Clicker Heroes',
 'Crab Game',
 'Hades',
 'Factorio',
 'Undertale',
 'Slay the Spire',
 'Dead Cells',
 'Cuphead'

In [6]:
import pandas as pd
import numpy as np


def preprocess_steam_indie(df):
    df = df.copy()

    # -------------------------
    # 1) 기본 문자열 정리
    # -------------------------
    text_cols = ["name_spy", "name_store", "developers", "publishers", "release_date"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # -------------------------
    # 2) 수치형 컬럼 변환
    # -------------------------
    numeric_cols = ["positive", "negative", "price_spy", "initialprice", "discount", "ccu"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # -------------------------
    # 3) owners 범위 문자열 처리
    # 예: "20,000 .. 50,000"
    # -------------------------
    if "owners" in df.columns:
        df["owners"] = df["owners"].astype("string").str.replace(",", "", regex=False)

        df["owners_low"] = df["owners"].str.split(r"\.\.").str[0].str.strip()
        df["owners_high"] = df["owners"].str.split(r"\.\.").str[1].str.strip()

        df["owners_low"] = pd.to_numeric(df["owners_low"], errors="coerce")
        df["owners_high"] = pd.to_numeric(df["owners_high"], errors="coerce")

        df["owners_mid"] = (df["owners_low"] + df["owners_high"]) / 2

    # -------------------------
    # 4) 가격 단위 변환
    # SteamSpy 가격은 보통 센트 단위일 가능성이 높음
    # -------------------------
    if "price_spy" in df.columns:
        df["price_usd"] = df["price_spy"] / 100

    if "initialprice" in df.columns:
        df["initialprice_usd"] = df["initialprice"] / 100

    # -------------------------
    # 5) 리뷰 관련 파생변수
    # -------------------------
    if {"positive", "negative"}.issubset(df.columns):
        df["review_total"] = df["positive"].fillna(0) + df["negative"].fillna(0)

        df["positive_ratio"] = np.where(
            df["review_total"] > 0,
            df["positive"] / df["review_total"],
            np.nan
        )

        df["negative_ratio"] = np.where(
            df["review_total"] > 0,
            df["negative"] / df["review_total"],
            np.nan
        )

        df["log_review_total"] = np.log1p(df["review_total"])

    # -------------------------
    # 6) 할인 여부
    # -------------------------
    if "discount" in df.columns:
        df["has_discount"] = np.where(df["discount"].fillna(0) > 0, 1, 0)

    # -------------------------
    # 7) 무료 게임 여부 정리
    # -------------------------
    if "is_free" in df.columns:
        df["is_free"] = df["is_free"].astype("boolean")

    # -------------------------
    # 8) 날짜 처리
    # -------------------------
    if "release_date" in df.columns:
        df["release_date_parsed"] = pd.to_datetime(df["release_date"], errors="coerce")

        # 기준일은 오늘 대신 분석 시점 고정값을 넣어도 됨
        today = pd.Timestamp.today().normalize()
        df["days_since_release"] = (today - df["release_date_parsed"]).dt.days

        df["release_year"] = df["release_date_parsed"].dt.year
        df["release_month"] = df["release_date_parsed"].dt.month

    # -------------------------
    # 9) 리스트 컬럼 문자열화
    # -------------------------
    list_cols = ["genres", "categories"]
    for col in list_cols:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: ", ".join(x) if isinstance(x, list) else x
            )

    # -------------------------
    # 10) 장르/카테고리 주요 플래그 만들기
    # -------------------------
    if "genres" in df.columns:
        df["has_action"] = df["genres"].astype("string").str.contains("Action", case=False, na=False).astype(int)
        df["has_rpg"] = df["genres"].astype("string").str.contains("RPG", case=False, na=False).astype(int)
        df["has_strategy"] = df["genres"].astype("string").str.contains("Strategy", case=False, na=False).astype(int)
        df["has_simulation"] = df["genres"].astype("string").str.contains("Simulation", case=False, na=False).astype(int)
        df["has_adventure"] = df["genres"].astype("string").str.contains("Adventure", case=False, na=False).astype(int)

    if "categories" in df.columns:
        df["is_singleplayer"] = df["categories"].astype("string").str.contains("Single-player", case=False, na=False).astype(int)
        df["is_multiplayer"] = df["categories"].astype("string").str.contains("Multi-player", case=False, na=False).astype(int)
        df["has_coop"] = df["categories"].astype("string").str.contains("Co-op", case=False, na=False).astype(int)
        df["has_controller"] = df["categories"].astype("string").str.contains("Controller", case=False, na=False).astype(int)

    # -------------------------
    # 11) 가격 구간화
    # -------------------------
    if "price_usd" in df.columns:
        df["price_band"] = pd.cut(
            df["price_usd"],
            bins=[-1, 0, 5, 10, 20, 30, 9999],
            labels=["Free", "0~5", "5~10", "10~20", "20~30", "30+"]
        )

    # -------------------------
    # 12) 리뷰 규모 구간화
    # -------------------------
    if "review_total" in df.columns:
        df["review_band"] = pd.cut(
            df["review_total"],
            bins=[-1, 10, 50, 100, 500, 1000, 999999999],
            labels=["0~10", "11~50", "51~100", "101~500", "501~1000", "1000+"]
        )

    return df

In [2]:
import requests
import pandas as pd
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def get_indie_apps_from_steamspy(pages=5):
    url = "https://steamspy.com/api.php"
    all_rows = []

    for page in range(pages):
        params = {
            "request": "genre",
            "genre": "Indie",
            "page": page
        }

        print(f"Indie {page} 페이지 수집 중...")

        res = requests.get(url, params=params, headers=HEADERS, timeout=30)
        res.raise_for_status()
        data = res.json()

        for appid_str, info in data.items():
            try:
                appid = int(appid_str)
            except:
                continue

            row = {
                "appid": appid,
                "name": info.get("name"),
                "owners": info.get("owners"),
                "positive": info.get("positive"),
                "negative": info.get("negative"),
                "price": info.get("price"),
                "initialprice": info.get("initialprice"),
                "discount": info.get("discount"),
                "ccu": info.get("ccu"),
            }
            all_rows.append(row)

        time.sleep(1)

    return pd.DataFrame(all_rows)


def get_store_app_details(appid):
    url = "https://store.steampowered.com/api/appdetails"
    params = {
        "appids": appid,
        "l": "english"
    }

    try:
        res = requests.get(url, params=params, headers=HEADERS, timeout=30)
        res.raise_for_status()

        raw = res.json()
        app_block = raw.get(str(appid), {})

        if not app_block.get("success"):
            return None

        data = app_block.get("data", {})

        genres = [g.get("description") for g in data.get("genres", []) if "description" in g]
        categories = [c.get("description") for c in data.get("categories", []) if "description" in c]

        return {
            "appid": appid,
            "name_store": data.get("name"),
            "type": data.get("type"),
            "genres": genres,
            "categories": categories,
            "release_date": data.get("release_date", {}).get("date"),
            "is_free": data.get("is_free"),
            "developers": ", ".join(data.get("developers", [])) if data.get("developers") else None,
            "publishers": ", ".join(data.get("publishers", [])) if data.get("publishers") else None,
        }

    except Exception as e:
        print(f"appid={appid} 조회 실패: {e}")
        return None


def is_indie_game(genres):
    if isinstance(genres, list):
        return "Indie" in genres
    if isinstance(genres, str):
        return "Indie" in genres
    return False


# 1) SteamSpy에서 Indie 장르 게임 직접 수집
df_apps = get_indie_apps_from_steamspy(pages=5)

print("SteamSpy Indie 후보 수:", len(df_apps))
print(df_apps.head())
print(df_apps.columns.tolist())

# 중복 제거
df_apps = df_apps.drop_duplicates(subset=["appid"]).copy()
print("중복 제거 후 후보 수:", len(df_apps))

# 2) 테스트용으로 200개만 상세 조회
sample_appids = df_apps["appid"].dropna().astype(int).head(200)

details = []
for i, appid in enumerate(sample_appids, start=1):
    item = get_store_app_details(appid)

    if i <= 5:
        print(f"appid={appid}, item={item}")

    if item is not None:
        details.append(item)

    if i % 20 == 0:
        print(f"{i}개 조회 완료 / 성공 {len(details)}개")

    time.sleep(1)

df_detail = pd.DataFrame(details)

print("details 개수:", len(details))
print("df_detail shape:", df_detail.shape)
print("df_detail columns:", df_detail.columns.tolist())
print(df_detail.head())

if df_detail.empty:
    print("상세 조회 결과가 비어 있습니다.")
else:
    # game 타입만
    if "type" in df_detail.columns:
        df_detail = df_detail[df_detail["type"] == "game"].copy()

    # Store 기준으로 최종 Indie 재검증
    if "genres" in df_detail.columns:
        df_indie = df_detail[df_detail["genres"].apply(is_indie_game)].copy()
        print(df_indie.head())
        print("최종 인디 게임 수:", len(df_indie))
    else:
        print("genres 컬럼이 없습니다.")

Indie 0 페이지 수집 중...
Indie 1 페이지 수집 중...
Indie 2 페이지 수집 중...


KeyboardInterrupt: 

In [1]:
df_clean = preprocess_steam_indie(df_final)

print(df_clean.head())
print(df_clean.columns.tolist())
print(df_clean.shape)

NameError: name 'preprocess_steam_indie' is not defined

In [ ]:
df_clean.to_csv("steam_indie_games_cleaned.csv", index=False, encoding="utf-8-sig")
print("저장 완료: steam_indie_games_cleaned.csv")

저장 완료: steam_indie_games_cleaned.csv
